 &nbsp; 

# MARIO
GTAP Power 2023 · 163 → 13 regions · 76 → 143 sectors

In [ ]:
from pathlib import Path
import json, os, sys, tempfile
from IPython.display import FileLink, display

cfg = json.loads(Path("disaggregation.json").read_text())
sys.path[:0] = [cfg["mario_repository"], str(Path(cfg["disaggregation_repository"]) / "src")]
os.environ["ENTICE_DATA_ROOT"] = cfg["data_root"]

In [ ]:
import mario
from mario.ops.add_sector_steps import load_checkpoint
from entice_inventory import Geography, SplitPlan, prepare_inputs, analysis, reporting

mario.set_log_verbosity("warning")
REBUILD = False
geography = Geography.read(cfg["geography"])
plan = SplitPlan.read(cfg["workbook"])
run_dir = Path(tempfile.mkdtemp(prefix="notebook-", dir=cfg["work_directory"]))

## 1. Database
Load GTAP-MRIO and aggregate its 163 regions into the 13 ENTICE regions.

In [ ]:
if REBUILD:
    db = mario.parse_from_parquet(cfg["native_database"], table="IOT", mode="flows")
    db.aggregate(None, levels="Region", region_aggregation=geography.mapping,
                 zero_output_epsilon=None)
else:
    db = mario.parse_from_parquet(cfg["baseline_database"], table="IOT", mode="flows")

In [ ]:
db

## 2. Supporting data
Aggregate production and trade totals, and combine cost structures using production weights.

In [ ]:
if REBUILD:
    prepared = prepare_inputs(cfg["native_inventories"], run_dir / "prepared_inputs",
        geography=geography, steps=plan.steps,
        baseline_path=cfg["native_database"], baseline_format="mario_parquet",
        cleaning_policy=True, output_reference_dir=cfg["output_references"])

## 3. Excel template
Generate the workbook used to define the new sectors and their supporting data.

In [ ]:
db.get_add_sectors_excel(str(run_dir / "add_sectors_blank.xlsx"),
                        items=list(plan.sectors), intermediate_steps=True, overwrite=True)

## 4. Input data
Read the reviewed workbook containing recipes, production targets and bilateral trade observations.

In [ ]:
display(FileLink(cfg["source_inventory"]), FileLink(cfg["workbook"]))
db.read_add_sectors_excel(cfg["workbook"], read_inventories=True, split=True)

## 5. Disaggregation
Expand the database from 76 to 143 sectors while preserving its aggregate monetary flows.

In [ ]:
if REBUILD:
    result = db.add_sectors("inventories", split=True, inplace=False,
        intermediate_steps=True, save_intermediates=run_dir / "checkpoints",
        solver=cfg["solver"], solver_parameters=cfg["solver_parameters"],
        split_model_options=cfg["split_model_options"])
else:
    result = load_checkpoint(cfg["result_checkpoint"])

In [ ]:
result

## 6. Results
Compare production, trade and product uses. The heatmap sums all origins and normalises each destination column to 100%.

In [ ]:
checks = analysis.check_targets(result, cfg["workbook"])
references = analysis.ReferenceSet.read(cfg["reference_set"]).compare(result)
dashboard = reporting.write_dashboard(result, "results/MARIO_workshop_dashboard.html",
    checks=checks, references=references, title="From 76 to 143 sectors with MARIO",
    subtitle="13 regions · GTAP Power 2023")
display(FileLink(str(dashboard)))

profile = analysis.use_profile(result, "APP")
analysis.plot_use_profile(profile, labels=analysis.sector_labels()).show()